# Page-classifier runtime benchmark — path template

This is the portable version of the V2 page-classifier benchmark. Replace every `xxx` path in the configuration cell before running it. No project-specific absolute path is embedded in this notebook.

## 1. Configuration

Use `sample` for a limited benchmark or `full` for every PDF in the input directory.

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import sys

import numpy as np
import pandas as pd

RUN_MODE = 'sample'
SAMPLE_PDF_COUNT = 100
RANDOM_SEED = 42

# TODO: Fill in the actual directory that contains page_classifier.py.
CLASSIFIER_CODE_DIR = Path('xxx')

# TODO: Fill in the actual directory that contains the input PDF files.
PDF_INPUT_DIR = Path('xxx')

# TODO: Fill in the actual path to enriched_training_cross_validation.csv.
TRAINING_CSV_PATH = Path('xxx')

# TODO: Fill in the actual directory where benchmark results should be saved.
OUTPUT_DIR = Path('xxx')

if RUN_MODE not in {'sample', 'full'}:
    raise ValueError("RUN_MODE must be 'sample' or 'full'.")
if not CLASSIFIER_CODE_DIR.is_dir():
    raise FileNotFoundError(CLASSIFIER_CODE_DIR)
if not PDF_INPUT_DIR.is_dir():
    raise FileNotFoundError(PDF_INPUT_DIR)
if not TRAINING_CSV_PATH.is_file():
    raise FileNotFoundError(TRAINING_CSV_PATH)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CLASSIFIER_CODE_DIR.resolve()))
from page_classifier import build_model_bundle, classify_pdf

print('Run mode:', RUN_MODE)
print('Classifier code:', CLASSIFIER_CODE_DIR)
print('PDF input:', PDF_INPUT_DIR)
print('Training table:', TRAINING_CSV_PATH)
print('Output:', OUTPUT_DIR)

## 2. Reconstruct V2 and select the benchmark PDFs

The benchmark reconstructs V2 from the saved feature table. This keeps training time visible and separate from PDF-processing time.

In [ ]:
training_start = perf_counter()
model_bundle = build_model_bundle(TRAINING_CSV_PATH)
training_seconds = perf_counter() - training_start

all_pdf_paths = sorted(PDF_INPUT_DIR.glob('*.pdf'))
if not all_pdf_paths:
    raise FileNotFoundError(f'No PDF files found in {PDF_INPUT_DIR}')

if RUN_MODE == 'full':
    benchmark_pdf_paths = all_pdf_paths
else:
    rng = np.random.default_rng(RANDOM_SEED)
    sample_size = min(SAMPLE_PDF_COUNT, len(all_pdf_paths))
    selected = np.sort(rng.choice(len(all_pdf_paths), size=sample_size, replace=False))
    benchmark_pdf_paths = [all_pdf_paths[int(index)] for index in selected]

print(f'Model reconstruction: {training_seconds:.3f} seconds')
print('Selected PDFs:', len(benchmark_pdf_paths))

## 3. Run the timed PDF benchmark

In [ ]:
prediction_frames = []
timing_rows = []
processing_start = perf_counter()

for index, pdf_path in enumerate(benchmark_pdf_paths, start=1):
    pdf_start = perf_counter()
    frame = classify_pdf(pdf_path, model_bundle)
    elapsed = perf_counter() - pdf_start
    prediction_frames.append(frame)
    timing_rows.append({
        'pdf': pdf_path.name,
        'pages': int(len(frame)),
        'seconds': elapsed,
        'seconds_per_page': elapsed / max(len(frame), 1),
    })
    if index % 10 == 0 or index == len(benchmark_pdf_paths):
        print(f'Processed {index}/{len(benchmark_pdf_paths)} PDFs')

processing_seconds = perf_counter() - processing_start
predictions = pd.concat(prediction_frames, ignore_index=True)
pdf_timings = pd.DataFrame(timing_rows)

predictions_path = OUTPUT_DIR / f'{RUN_MODE}_predictions.csv'
timings_path = OUTPUT_DIR / f'{RUN_MODE}_pdf_timings.csv'
predictions.to_csv(predictions_path, index=False)
pdf_timings.to_csv(timings_path, index=False)

summary = {
    'run_mode': RUN_MODE,
    'pdf_count': int(len(benchmark_pdf_paths)),
    'page_count': int(len(predictions)),
    'training_seconds': float(training_seconds),
    'processing_seconds': float(processing_seconds),
    'seconds_per_pdf': float(processing_seconds / len(benchmark_pdf_paths)),
    'seconds_per_page': float(processing_seconds / len(predictions)),
    'predicted_text_pages': int((predictions.predicted_page_type == 'text').sum()),
    'predicted_drawing_pages': int((predictions.predicted_page_type == 'drawing').sum()),
}
summary_path = OUTPUT_DIR / f'{RUN_MODE}_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(json.dumps(summary, indent=2))
print('Saved predictions:', predictions_path)
print('Saved timings:', timings_path)
print('Saved summary:', summary_path)

## Interpretation

- `training_seconds` measures V2 reconstruction from saved features.
- `processing_seconds` includes PDF rendering, feature extraction, and prediction.
- Use the Python script's `train` command once to save a joblib model when normal prediction must avoid all future training.